In [ ]:
import os
import pandas as pd
import numpy as np
from feedbackFunctions import get_paths, extract_timeseries, read_events, iteration

In [ ]:
# Settings
data_dir = '/Users/alexandresayal/sftp/DATAPOOL/VPMB/BIDS-VPMB-SPE'
fmriprep_dir = '/Users/alexandresayal/sftp/DATAPOOL/VPMB/BIDS-VPMB-SPE/derivatives/fmriprep23/fmriprep'
nilearn_dir = '/Users/alexandresayal/sftp/DATAPOOL/VPMB/BIDS-VPMB-SPE/derivatives/nilearn_glm'
output_dir = '/Users/alexandresayal/sftp/DATAPOOL/VPMB/BIDS-VPMB-SPE/derivatives/feedbackSimulator'

subject_list = [x for x in os.listdir(data_dir) if 'sub-' in x]
subject_list.sort()

tr_list = [0.5, 0.75, 1, 2.5]
n_volumes_list = [780, 520, 390, 156]
run_list = ['AA','UA']
hrf_delay = 4 # in seconds

In [ ]:
# Load subject-specific roi coordinates
roi_ss_coords = pd.read_csv(os.path.join(nilearn_dir,'group','roi_ss_matrix.txt'), sep='\t', header=None,
                            names=['left_x','left_y','left_z','right_x','right_y','right_z'])

# add new column with the subject names
roi_ss_coords['subject'] = subject_list
labels = ['hMT+ L', 'hMT+ R']

In [ ]:
# Initialize matrix to store the timeseries of all subjects per tr
TC = np.zeros((len(subject_list), len(tr_list), len(run_list), max(n_volumes_list)))


In [ ]:
# Run in parallel
from joblib import Parallel, delayed

Parallel(n_jobs=8)(
    delayed(
        iteration(ss,subject, tr_list, run_list, n_volumes_list, roi_ss_coords, hrf_delay, fmriprep_dir, data_dir, TC)
        )
        (ss,subject) for ss,subject in enumerate(subject_list)
        )


